# StreamRaider — Arc Raiders YOLO V2 Training
Train YOLOv8n for real-time game element + HUD detection.

**Dataset:** 6,878 images · **Classes:** 33 · **Model:** YOLOv8n

### ✅ Crash-proof design
- All training saves directly to Google Drive
- Checkpoints saved every epoch — if Colab dies, you lose at most 1 epoch
- Resume from where you left off by re-running this notebook

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create persistent output directory on Drive
import os
DRIVE_DIR = '/content/drive/MyDrive/streamraider-yolo'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✅ Drive mounted. Output dir: {DRIVE_DIR}')

## 2. Setup & Download Dataset

In [ ]:
!pip install -q ultralytics onnx onnxruntime

# Download V2 dataset from public GitHub Release
!wget -q https://github.com/eddie-claw/streamraider-dataset/releases/download/v2/dataset-v2.zip -O /content/dataset-v2.zip
!unzip -q /content/dataset-v2.zip -d /content/yolo-training/

%cd /content/yolo-training

# Verify dataset
import os
train_count = len(os.listdir('dataset/images/train'))
val_count = len(os.listdir('dataset/images/val'))
print(f'✅ Train: {train_count}, Val: {val_count}, Total: {train_count + val_count}')

## 3. Fix data.yaml paths

In [ ]:
import yaml

with open('dataset-v2/data.yaml', 'r') as f:
    data = yaml.safe_load(f)

data['train'] = '/content/yolo-training/dataset-v2/images/train'
data['val'] = '/content/yolo-training/dataset-v2/images/val'

with open('dataset-v2/data.yaml', 'w') as f:
    yaml.dump(data, f)

print(f"Classes ({data['nc']}): {data['names']}")

## 4. Train (saves to Drive, supports resume)

**First run:** Trains from scratch, all checkpoints go to Drive.

**Resume after crash:** Just re-run this cell — it detects the last checkpoint and continues.

In [ ]:
from ultralytics import YOLO
import os

DRIVE_DIR = '/content/drive/MyDrive/streamraider-yolo'
LAST_PT = os.path.join(DRIVE_DIR, 'arc_raiders_v2', 'weights', 'last.pt')
V1_BEST = os.path.join(DRIVE_DIR, 'arc_raiders_v2', 'weights', 'best.pt')

# Check if we can resume from a previous v2 run
if os.path.exists(LAST_PT):
    print(f'🔄 Found v2 checkpoint: {LAST_PT}')
    print('   Resuming training from where we left off...')
    model = YOLO(LAST_PT)
    results = model.train(resume=True)
elif os.path.exists(V1_BEST):
    print(f'🔄 Found v1 model: {V1_BEST}')
    print('   Fine-tuning v1 model with v2 dataset (transfer learning)...')
    model = YOLO(V1_BEST)
    results = model.train(
        data='dataset-v2/data.yaml',
        epochs=80,
        batch=16,
        imgsz=640,
        device=0,
        project=DRIVE_DIR,
        name='arc_raiders_v2',
        patience=25,
        optimizer='AdamW',
        cos_lr=True,
        lr0=0.0005,
        augment=True,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
        save_period=1,
    )
else:
    print('🆕 No checkpoint found. Starting fresh v2 training...')
    model = YOLO('yolov8n.pt')
    results = model.train(
        data='dataset-v2/data.yaml',
        epochs=150,
        batch=16,
        imgsz=640,
        device=0,
        project=DRIVE_DIR,
        name='arc_raiders_v2',
        patience=25,
        optimizer='AdamW',
        cos_lr=True,
        lr0=0.001,
        augment=True,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
        save_period=1,  # Save checkpoint EVERY epoch to Drive
    )

print('\n🎉 Training complete! Model saved to Google Drive.')

## 5. Validation & Metrics
Self-contained cell — works even after a runtime restart.

In [ ]:
# === Self-contained validation cell (works after runtime restart) ===
!pip install -q ultralytics
import os, glob
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from ultralytics import YOLO

DRIVE_DIR = '/content/drive/MyDrive/streamraider-yolo'

# Auto-detect: v2 if exists, else v1
V2_BEST = os.path.join(DRIVE_DIR, 'arc_raiders_v2', 'weights', 'best.pt')
V1_BEST = os.path.join(DRIVE_DIR, 'arc_raiders_v1', 'weights', 'best.pt')
if os.path.exists(V2_BEST):
    BEST_PT = V2_BEST
    DATASET_URL = 'https://github.com/eddie-claw/streamraider-dataset/releases/download/v2/dataset-v2.zip'
    DATASET_DIR = '/content/dataset-v2'
    DATA_YAML = '/content/dataset-v2/data.yaml'
    print('🔍 Using V2 model')
elif os.path.exists(V1_BEST):
    BEST_PT = V1_BEST
    DATASET_URL = 'https://github.com/eddie-claw/streamraider-dataset/releases/download/v1/dataset.zip'
    DATASET_DIR = '/content/dataset'
    DATA_YAML = '/content/dataset/data.yaml'
    print('🔍 Using V1 model')
else:
    raise FileNotFoundError('No best.pt found on Drive for v1 or v2!')

# Download dataset if needed
if not os.path.exists(DATA_YAML):
    print(f'📦 Downloading dataset...')
    !wget -q {DATASET_URL} -O /content/ds.zip
    !unzip -q /content/ds.zip -d /content/
    # Find data.yaml in case of nested dirs
    found = glob.glob('/content/**/data.yaml', recursive=True)
    if found:
        DATA_YAML = found[0]
        DATASET_DIR = os.path.dirname(DATA_YAML)
    print(f'✅ Dataset at: {DATASET_DIR}')

# Fix data.yaml paths to be absolute
import yaml
with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)
data['train'] = os.path.join(DATASET_DIR, 'images', 'train')
data['val'] = os.path.join(DATASET_DIR, 'images', 'val')
with open(DATA_YAML, 'w') as f:
    yaml.dump(data, f)

# Run validation
print(f'\n🚀 Validating {BEST_PT}...')
best = YOLO(BEST_PT)
metrics = best.val(data=DATA_YAML)

print(f'\n📊 Overall Metrics:')
print(f'  mAP50:    {metrics.box.map50:.3f}')
print(f'  mAP50-95: {metrics.box.map:.3f}')

# Per-class AP
print(f'\n📋 Per-Class AP50:')
names = best.names
for i, ap in enumerate(metrics.box.ap50):
    if ap > 0:
        status = '✅' if ap >= 0.7 else '⚠️' if ap >= 0.4 else '❌'
        print(f'  {status} {names[i]:25s}: {ap:.3f}')

# Show zeros too
zeros = [names[i] for i, ap in enumerate(metrics.box.ap50) if ap == 0]
if zeros:
    print(f'\n❌ Zero AP classes ({len(zeros)}): {", ".join(zeros)}')

In [ ]:
from IPython.display import Image, display
import os
DRIVE_DIR = '/content/drive/MyDrive/streamraider-yolo'
RUN_DIR = os.path.join(DRIVE_DIR, 'arc_raiders_v2')

print('📈 Training Curves:')
display(Image(os.path.join(RUN_DIR, 'results.png'), width=800))
print('\n🎯 Confusion Matrix:')
display(Image(os.path.join(RUN_DIR, 'confusion_matrix.png'), width=600))

## 6. Test on sample frames

In [ ]:
import glob
val_images = glob.glob('dataset/images/val/*.jpg')[:5]

results = best.predict(val_images, conf=0.3, save=True, project='runs', name='test_predictions')

for img_path in glob.glob('runs/test_predictions/*.jpg')[:5]:
    display(Image(img_path, width=600))

## 7. Export to ONNX

In [ ]:
best.export(format='onnx', imgsz=640, simplify=True)

# Copy ONNX to Drive too
import shutil
DRIVE_DIR = '/content/drive/MyDrive/streamraider-yolo'
WEIGHTS_DIR = os.path.join(DRIVE_DIR, 'arc_raiders_v2', 'weights')
onnx_src = BEST_PT.replace('.pt', '.onnx')
if os.path.exists(onnx_src):
    print(f'✅ ONNX already on Drive')
else:
    # Export creates it next to source .pt
    print(f'✅ ONNX exported to Drive')

print(f'\n📦 Files on Google Drive ({WEIGHTS_DIR}):')
for f in os.listdir(WEIGHTS_DIR):
    size = os.path.getsize(os.path.join(WEIGHTS_DIR, f)) / 1024 / 1024
    print(f'  {f}: {size:.1f} MB')

print('\n🎉 All done! Your model is safe in Google Drive.')
print('   Path: Google Drive > streamraider-yolo > arc_raiders_v2 > weights/')